In [4]:
import cv2
import torch
import torch.nn.functional as F

def nothing(x):
    pass

# Kamera öffnen (meistens 0; wenn du mehrere Kameras hast, evtl. 1, 2, ...)
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Kamera konnte nicht geöffnet werden")
else:
    window_name = "Thresholding (PyTorch)"
    cv2.namedWindow(window_name)

    # Initialwerte für Trackbars
    # Kernel Size: Start 15, Max 101
    cv2.createTrackbar("Kernel", window_name, 15, 101, nothing)
    # C: Start 5 (entspricht 0.05), Max 100 (entspricht 1.0)
    cv2.createTrackbar("C (*100)", window_name, 5, 100, nothing)
    # Global Threshold: Start 127, Max 255
    cv2.createTrackbar("Global Thresh", window_name, 127, 255, nothing)

    # "Buttons" als Trackbars (0=Aus, 1=An) simulieren, da echte Buttons Qt benötigen
    # Mode: 0 = Global, 1 = Dynamic
    cv2.createTrackbar("Mode (0:Glob, 1:Dyn)", window_name, 1, 1, nothing)
    # Invert: 0 = Off, 1 = On
    cv2.createTrackbar("Invert (0:Off, 1:On)", window_name, 0, 1, nothing)

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Werte von Trackbars lesen
        k_val = cv2.getTrackbarPos("Kernel", window_name)
        c_val = cv2.getTrackbarPos("C (*100)", window_name)
        g_val = cv2.getTrackbarPos("Global Thresh", window_name)
        
        # Status aus "Button"-Trackbars lesen
        mode_pos = cv2.getTrackbarPos("Mode (0:Glob, 1:Dyn)", window_name)
        use_dynamic = (mode_pos == 1)
        
        invert_pos = cv2.getTrackbarPos("Invert (0:Off, 1:On)", window_name)
        invert_colors = (invert_pos == 1)

        # BGR -> Gray (OpenCV nutzt BGR)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Nach [0,1] normalisieren und in Tensor umwandeln: Form (1, 1, H, W)
        img = torch.from_numpy(gray).float() / 255.0
        img = img.unsqueeze(0).unsqueeze(0)  # (N=1, C=1, H, W)

        if use_dynamic:
            # Kernel Size muss ungerade und > 0 sein
            if k_val % 2 == 0:
                k_val += 1
            if k_val < 1:
                k_val = 1
            
            kernel_size = k_val
            C = c_val / 100.0

            # Kernel für lokalen Mittelwert (Box-Filter) neu erstellen
            kernel = torch.ones(1, 1, kernel_size, kernel_size, dtype=torch.float32)
            kernel = kernel / kernel.numel()  # Mittelwert statt Summe

            # Lokalen Mittelwert mit conv2d berechnen
            mean_local = F.conv2d(img, kernel, padding=kernel_size // 2)

            # Dynamischer Schwellwert: 1 wenn Pixel < (Mittelwert - C), sonst 0
            binary = (img < (mean_local - C)).float()
        else:
            # Globaler Schwellwert
            binary = (img > (g_val / 255.0)).float()

        # Zurück nach NumPy, in [0,255] skalieren und uint8 für Anzeige
        out = (binary.squeeze(0).squeeze(0).numpy() * 255).astype("uint8")

        # Farben umkehren
        if invert_colors:
            out = cv2.bitwise_not(out)

        # Modus anzeigen
        mode_str = "Dynamic" if use_dynamic else "Global"
        cv2.putText(out, f"Mode: {mode_str}", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, 127, 2)

        # Original und gefiltertes Bild anzeigen
        cv2.imshow("Original", gray)
        cv2.imshow(window_name, out)

        # Tastenabfrage
        key = cv2.waitKey(1) & 0xFF
        if key == 27: # ESC zum Beenden
            break
        elif key == ord('i'): # 'i' zum Invertieren (aktualisiert auch Trackbar)
            new_val = 0 if invert_colors else 1
            cv2.setTrackbarPos("Invert (0:Off, 1:On)", window_name, new_val)
        elif key == ord('m'): # 'm' zum Modus wechseln (aktualisiert auch Trackbar)
            new_val = 0 if use_dynamic else 1
            cv2.setTrackbarPos("Mode (0:Glob, 1:Dyn)", window_name, new_val)

    cap.release()
    cv2.destroyAllWindows()

In [5]:
import cv2
import numpy as np

# Kamera öffnen
cap = cv2.VideoCapture(1)

if not cap.isOpened():
    print("Kamera konnte nicht geöffnet werden")
else:
    roi_hist = None
    
    print("Drücke 's', um eine Region of Interest (ROI) auszuwählen.")
    print("Drücke 'r', um die Auswahl zurückzusetzen.")
    print("Drücke 'ESC', um zu beenden.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if roi_hist is None:
            # Solange kein ROI gewählt ist, zeige das Originalbild mit Hinweis
            display_frame = frame.copy()
            cv2.putText(display_frame, "Druecke 's' fuer ROI Auswahl", (10, 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.imshow("Kamera", display_frame)
            
            key = cv2.waitKey(1) & 0xFF
            if key == ord('s'):
                # ROI Auswahl starten
                # selectROI gibt (x, y, w, h) zurück. Das Fenster friert kurz ein für die Auswahl.
                roi_rect = cv2.selectROI("Kamera", frame, fromCenter=False, showCrosshair=True)
                x, y, w, h = roi_rect
                
                if w > 0 and h > 0:
                    # ROI ausschneiden und verarbeiten
                    roi = frame[y:y+h, x:x+w]
                    roi_lab = cv2.cvtColor(roi, cv2.COLOR_BGR2Lab)
                    
                    # Histogramm berechnen (nur a und b Kanäle)
                    roi_ab = roi_lab[:, :, 1:3]
                    roi_hist = cv2.calcHist([roi_ab], [0, 1], None, [32, 32], [0, 256, 0, 256])
                    cv2.normalize(roi_hist, roi_hist, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX)
                else:
                    print("Ungültige Auswahl")

        else:
            # Wenn ROI gewählt ist, führe Backprojection durch
            img_lab = cv2.cvtColor(frame, cv2.COLOR_BGR2Lab)
            img_ab = img_lab[:, :, 1:3]
            
            # Backprojection
            backproj = cv2.calcBackProject([img_ab], [0, 1], roi_hist, [0, 256, 0, 256], scale=1)
            
            # Glätten und Schwellwert
            # Disc-Kernel für Glättung der Backprojection
            disc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
            cv2.filter2D(backproj, -1, disc, backproj)
            
            # Binäre Maske erstellen
            _, mask = cv2.threshold(backproj, 50, 255, cv2.THRESH_BINARY)
            
            # Maske für Anzeige auf 3 Kanäle erweitern
            mask_3ch = cv2.merge((mask, mask, mask))
            
            # Maske anwenden
            result = cv2.bitwise_and(frame, mask_3ch)
            
            cv2.imshow("Kamera", frame)
            cv2.imshow("Backprojection", backproj)
            cv2.imshow("Maske", mask)
            cv2.imshow("Ergebnis", result)
            
            key = cv2.waitKey(1) & 0xFF
            if key == ord('r'):
                roi_hist = None
                cv2.destroyWindow("Backprojection")
                cv2.destroyWindow("Maske")
                cv2.destroyWindow("Ergebnis")

        if key == 27: # ESC
            break

    cap.release()
    cv2.destroyAllWindows()

Drücke 's', um eine Region of Interest (ROI) auszuwählen.
Drücke 'r', um die Auswahl zurückzusetzen.
Drücke 'ESC', um zu beenden.


In [7]:
import cv2
import numpy as np
from collections import deque

def region_growing(gray_img, seed, threshold=5):
    """
    gray_img: 2D NumPy-Array (uint8)
    seed: (y, x)
    threshold: erlaubte Abweichung vom Seed-Wert
    """
    h, w = gray_img.shape
    sy, sx = seed
    
    # Sicherstellen, dass Seed innerhalb des Bildes liegt
    if not (0 <= sy < h and 0 <= sx < w):
        return np.zeros_like(gray_img, dtype=np.uint8)
        
    seed_value = gray_img[sy, sx]

    # Maske für Region (0 = kein Mitglied, 1 = Mitglied)
    region = np.zeros_like(gray_img, dtype=np.uint8)

    # Queue für BFS
    q = deque()
    q.append((sy, sx))
    region[sy, sx] = 1

    # 4er-Nachbarschaft
    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]

    while q:
        y, x = q.popleft()
        for dy, dx in neighbors:
            ny, nx = y + dy, x + dx
            if 0 <= ny < h and 0 <= nx < w and region[ny, nx] == 0:
                # Bedingung: Pixelwert ähnlich dem Seed-Wert
                if abs(int(gray_img[ny, nx]) - int(seed_value)) <= threshold:
                    region[ny, nx] = 1
                    q.append((ny, nx))

    return region

# Globale Variablen für Maus-Callback
seed_point = None

def mouse_callback(event, x, y, flags, param):
    global seed_point
    if event == cv2.EVENT_LBUTTONDOWN:
        # x ist Spalte, y ist Zeile -> seed erwartet (y, x)
        seed_point = (y, x)

# Kamera öffnen
cap = cv2.VideoCapture(1)
if not cap.isOpened():
    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        print("Keine Kamera gefunden")
else:
    cv2.namedWindow("Region Growing Live")
    cv2.setMouseCallback("Region Growing Live", mouse_callback)
    cv2.createTrackbar("Threshold", "Region Growing Live", 20, 100, lambda x: None)

    print("Klicke ins Bild, um den Startpunkt (Seed) zu setzen.")
    print("Passe den Threshold an.")
    print("Drücke 'r', um den Seed zu löschen.")
    print("ESC zum Beenden.")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        # Bild verkleinern für Performance (Region Growing in Python ist langsam)
        scale = 0.2
        small_frame = cv2.resize(frame, None, fx=scale, fy=scale)
        small_gray = cv2.cvtColor(small_frame, cv2.COLOR_BGR2GRAY)

        thresh_val = cv2.getTrackbarPos("Threshold", "Region Growing Live")

        if seed_point is not None:
            # Seed auf kleine Auflösung umrechnen
            sy, sx = seed_point
            small_seed = (int(sy * scale), int(sx * scale))
            
            # Region Growing ausführen
            mask_small = region_growing(small_gray, small_seed, thresh_val)
            
            # Maske wieder hochskalieren
            mask = cv2.resize(mask_small, (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST)
            
            # Overlay erstellen (Rot)
            overlay = np.zeros_like(frame)
            overlay[mask == 1] = [0, 0, 255]
            
            # Überlagern
            display = cv2.addWeighted(frame, 0.7, overlay, 0.3, 0)
            
            # Seed-Punkt markieren
            cv2.circle(display, (seed_point[1], seed_point[0]), 5, (0, 255, 0), -1)
        else:
            display = frame

        cv2.imshow("Region Growing Live", display)

        key = cv2.waitKey(1) & 0xFF
        if key == 27: # ESC
            break
        elif key == ord('r'):
            seed_point = None

    cap.release()
    cv2.destroyAllWindows()

Klicke ins Bild, um den Startpunkt (Seed) zu setzen.
Passe den Threshold an.
Drücke 'r', um den Seed zu löschen.
ESC zum Beenden.
